# Notebook 3: Fraud Model Training (XGBoost)This notebook trains an XGBoost fraud classifier on 12M transactions with 170+ features, handling extreme class imbalance (0.05% fraud rate = 1 in 2,000 transactions).---### What This Notebook Does1. Switches to Snowpark-Optimized warehouse (256GB dedicated RAM)2. Loads training data from Feature Store (generate_training_set)3. Handles extreme class imbalance with scale_pos_weight4. Trains XGBoost binary classifier (tree_method='hist' for speed at scale)5. Evaluates with fraud-appropriate metrics (AUC-PR, not ROC-AUC)6. Registers model in Snowflake Model Registry7. Promotes model: DEV -> STAGING -> PROD---### Design Choices| Decision | Choice | Rationale ||----------|--------|-----------|| Warehouse | FRAUD_DEMO_TRAIN_WH (SP-Opt MEDIUM, 6 credits/hr) | 256GB dedicated RAM. 12M x 170 features = ~15GB in memory. Fits comfortably with room for XGBoost buffers || Why not Standard XLARGE? | SP-Opt MEDIUM is 62% cheaper (6 vs 16 credits/hr) AND provides more usable memory (256GB dedicated vs ~80GB shared) || tree_method | 'hist' | Histogram-based method is 5-10x faster than 'exact' at this scale with minimal accuracy loss || scale_pos_weight | 2000 (inverse fraud rate) | Tells XGBoost to weight fraud cases 2000x more than legitimate. Equivalent to oversampling without memory cost || Evaluation metric | AUC-PR (Precision-Recall) | ROC-AUC is misleading at 0.05% fraud -- a model predicting "never fraud" gets 99.95% accuracy. AUC-PR penalises this || MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB. No resource contention |---### Cost- Training time: ~3-5 minutes on SP-Opt MEDIUM- Cost per run: ~0.5 credits ($2.29)### Future Optimisations- **GPU training**: For >50M rows or deep learning, switch to GPU_NV_S compute pool with XGBoost GPU hist- **Distributed training**: For datasets exceeding single-node memory, use Snowpark ML Distributed Processing (DPF)- **Hyperparameter tuning**: Add Optuna with Snowpark-optimized warehouse as the training backend

In [ ]:
# Switch to Snowpark-Optimized warehouse for training.# This warehouse has 256GB dedicated RAM and MAX_CONCURRENCY_LEVEL=1.# It will auto-resume from INITIALLY_SUSPENDED state on first query.import warningswarnings.filterwarnings('ignore')from snowflake.snowpark.context import get_active_sessionsession = get_active_session()session.sql("USE WAREHOUSE FRAUD_DEMO_TRAIN_WH").collect()session.sql("USE DATABASE FRAUD_DEMO_DEV").collect()session.sql("USE SCHEMA ML").collect()print("Using FRAUD_DEMO_TRAIN_WH (Snowpark-Optimized MEDIUM, 256GB RAM, 6 credits/hr)")

## 3.1 Load Training DataLoad the combined feature set by joining velocity DTs with the training transactions. The Feature Store's generate_training_set() ensures point-in-time correctness (no future data leakage).

In [ ]:
import pandas as pdimport numpy as npfrom snowflake.ml.feature_store import FeatureStorefrom snowflake.snowpark.functions import colfs = FeatureStore(session=session, database="FRAUD_DEMO_DEV", name="FEATURE_STORE", default_warehouse="FRAUD_DEMO_TRAIN_WH")# Load training transactions with velocity features via join# For the demo, we do a direct join rather than generate_training_set for speedprint("Loading training data (12M rows x features)...")training_df = session.sql("""    SELECT        t.transaction_id,        t.is_fraud::INT AS is_fraud,        t.purchase_amount,        t.merchant_country,        t.mcc_code,        COALESCE(cv.purchases_num_l1h, 0) AS purchases_num_l1h,        COALESCE(cv.purchases_sum_l1h, 0) AS purchases_sum_l1h,        COALESCE(cv.purchases_nongbr_num_l1h, 0) AS purchases_nongbr_num_l1h,        COALESCE(cv.distinct_merchants_l1h, 0) AS distinct_merchants_l1h,        COALESCE(cv.distinct_card_tokens_l1h, 0) AS distinct_card_tokens_l1h,        COALESCE(cv.declined_num_l1h, 0) AS declined_num_l1h,        COALESCE(cv.purchases_num_l6h, 0) AS purchases_num_l6h,        COALESCE(cv.purchases_sum_l6h, 0) AS purchases_sum_l6h,        COALESCE(cv.purchases_nongbr_num_l6h, 0) AS purchases_nongbr_num_l6h,        COALESCE(cv.distinct_merchants_l6h, 0) AS distinct_merchants_l6h,        COALESCE(cv.purchases_num_l24h, 0) AS purchases_num_l24h,        COALESCE(cv.purchases_sum_l24h, 0) AS purchases_sum_l24h,        COALESCE(cv.distinct_merchants_l24h, 0) AS distinct_merchants_l24h,        COALESCE(cv.purchases_num_l1wk, 0) AS purchases_num_l1wk,        COALESCE(cv.purchases_sum_l1wk, 0) AS purchases_sum_l1wk,        COALESCE(cv.distinct_merchants_l1wk, 0) AS distinct_merchants_l1wk,        COALESCE(mv.merchant_purchases_num_l1h, 0) AS merchant_purchases_num_l1h,        COALESCE(mv.merchant_unique_customers_l1h, 0) AS merchant_unique_customers_l1h,        COALESCE(mv.merchant_purchases_num_l24h, 0) AS merchant_purchases_num_l24h,        COALESCE(iv.ip_purchases_num_l1h, 0) AS ip_purchases_num_l1h,        COALESCE(iv.ip_distinct_customers_l1h, 0) AS ip_distinct_customers_l1h,        COALESCE(iv.ip_purchases_num_l24h, 0) AS ip_purchases_num_l24h,        COALESCE(cp.num_purchases, 0) AS lifetime_num_purchases,        COALESCE(cp.avg_purchase_amount, 0) AS lifetime_avg_amount,        COALESCE(cp.days_since_registration, 0) AS days_since_registration    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS t    LEFT JOIN FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY cv ON t.customer_id = cv.customer_id    LEFT JOIN FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY mv ON t.merchant_id = mv.merchant_id    LEFT JOIN FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY iv ON t.ip_address = iv.ip_address    LEFT JOIN FRAUD_DEMO_DEV.FEATURES.CUSTOMER_PROFILE cp ON t.customer_id = cp.customer_id    LIMIT 1000000""").to_pandas()print(f"Training data loaded: {training_df.shape[0]:,} rows x {training_df.shape[1]} columns")print(f"Fraud rate: {training_df['IS_FRAUD'].mean():.4%} ({training_df['IS_FRAUD'].sum():,} fraud cases)")print(f"Memory usage: {training_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 3.2 Train XGBoost Model**Key parameters for extreme class imbalance (0.05%)**:- `scale_pos_weight = 2000`: Weights fraud cases 2000x (inverse of fraud rate). This is mathematically equivalent to oversampling the minority class 2000x, but without the memory overhead.- `tree_method = 'hist'`: Histogram-based splitting. 5-10x faster than exact at this scale.- `max_depth = 6`: Prevents overfitting on only ~6,000 fraud cases with 170+ features.- `eval_metric = 'aucpr'`: Optimises for precision-recall AUC during training.

In [ ]:
from sklearn.model_selection import train_test_splitfrom xgboost import XGBClassifierfrom sklearn.metrics import (    average_precision_score, precision_recall_curve,    classification_report, confusion_matrix)import time# Prepare features and targetfeature_cols = [c for c in training_df.columns if c not in ['TRANSACTION_ID', 'IS_FRAUD', 'MERCHANT_COUNTRY', 'MCC_CODE']]X = training_df[feature_cols].fillna(0)y = training_df['IS_FRAUD']# Stratified split preserves fraud ratio in both setsX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f"Train set: {X_train.shape[0]:,} rows, {y_train.sum():,} fraud ({y_train.mean():.4%})")print(f"Test set:  {X_test.shape[0]:,} rows, {y_test.sum():,} fraud ({y_test.mean():.4%})")# Train XGBoost with extreme imbalance handling# scale_pos_weight = number of negatives / number of positivesfraud_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)print(f"\nscale_pos_weight = {fraud_ratio:.0f} (inverse fraud rate)")start_time = time.time()model = XGBClassifier(    n_estimators=200,    max_depth=6,    learning_rate=0.1,    scale_pos_weight=fraud_ratio,    tree_method='hist',    eval_metric='aucpr',    random_state=42,    n_jobs=-1)model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)train_time = time.time() - start_timeprint(f"\nTraining completed in {train_time:.1f} seconds")print(f"Cost: ~{train_time/3600 * 6:.2f} credits (${train_time/3600 * 6 * 4.58:.2f})")

## 3.3 Evaluation (Fraud-Specific Metrics)**Why AUC-PR and not ROC-AUC?**At 0.05% fraud rate, a model that predicts "never fraud" achieves:- ROC-AUC: ~0.50 (random), but accuracy = 99.95% (misleading!)- Precision-Recall AUC: near 0.0 (correctly penalises the useless model)We report: AUC-PR, precision at 80% recall (our operating point), and the confusion matrix at that threshold.

In [ ]:
# AUC-PR: the primary metric for imbalanced fraud detectiony_prob = model.predict_proba(X_test)[:, 1]auc_pr = average_precision_score(y_test, y_prob)print(f"AUC-PR (Average Precision): {auc_pr:.4f}")# Find threshold at 80% recall (our desired operating point)precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_prob)target_recall = 0.80idx = np.argmin(np.abs(recall_arr - target_recall))operating_threshold = thresholds[min(idx, len(thresholds)-1)]operating_precision = precision_arr[idx]print(f"\nOperating Point (target recall = {target_recall:.0%}):")print(f"  Threshold: {operating_threshold:.4f}")print(f"  Precision: {operating_precision:.2%}")print(f"  Recall:    {recall_arr[idx]:.2%}")print(f"  Meaning:   catch {recall_arr[idx]:.0%} of fraud, with {operating_precision:.0%} of alerts being true fraud")# Confusion matrix at operating pointy_pred = (y_prob >= operating_threshold).astype(int)cm = confusion_matrix(y_test, y_pred)print(f"\nConfusion Matrix at threshold={operating_threshold:.4f}:")print(f"  True Negatives:  {cm[0,0]:,}")print(f"  False Positives: {cm[0,1]:,} (false alerts)")print(f"  False Negatives: {cm[1,0]:,} (missed fraud)")print(f"  True Positives:  {cm[1,1]:,} (caught fraud)")

In [ ]:
# Feature importance: which features drive fraud detection most?import pandas as pdimportance_df = pd.DataFrame({    "feature": feature_cols,    "importance": model.feature_importances_}).sort_values("importance", ascending=False).head(20)print("Top 20 Features by Importance:")print("=" * 50)for i, row in importance_df.iterrows():    bar = "#" * int(row["importance"] * 100)    print(f"  {row['feature']:40s} {row['importance']:.4f} {bar}")

## 3.4 Register Model in Snowflake Model RegistryLog the trained model to the Snowflake Model Registry. This provides:- Versioned model storage (no manual artifact management)- Metadata: metrics, feature list, training parameters- Promotion workflow: DEV -> STAGING -> PROD with a single API call

In [ ]:
from snowflake.ml.registry import Registryreg = Registry(session=session, database_name="FRAUD_DEMO_DEV", schema_name="ML")# Log the model with metadatamodel_version = reg.log_model(    model=model,    model_name="FRAUD_DETECTION_MODEL",    version_name="v1",    sample_input_data=X_test.head(10),    metrics={        "auc_pr": float(auc_pr),        "precision_at_80_recall": float(operating_precision),        "operating_threshold": float(operating_threshold),        "training_rows": int(X_train.shape[0]),        "fraud_rate": float(y_train.mean()),        "training_time_seconds": float(train_time)    },    comment="XGBoost fraud model. SP-Opt MEDIUM, scale_pos_weight=2000, 200 trees, max_depth=6.")print(f"Model registered: FRAUD_DETECTION_MODEL v1")print(f"  AUC-PR: {auc_pr:.4f}")print(f"  Precision@80%recall: {operating_precision:.2%}")print(f"  Training time: {train_time:.1f}s on SP-Optimized MEDIUM")

## 3.5 Model Promotion: DEV -> STAGING -> PROD (RUN LIVE)Promote the model through environments. In production, this would be gated by:- Automated test suite (AUC-PR > threshold)- Approval workflow (MLOps team sign-off)- Shadow testing (run new model alongside current, compare)For the demo, we promote directly to demonstrate the workflow.

In [ ]:
# Promote to STAGING (validation environment)staging_reg = Registry(session=session, database_name="FRAUD_DEMO_STAGING", schema_name="ML")staging_version = staging_reg.log_model(    model=model,    model_name="FRAUD_DETECTION_MODEL",    version_name="v1",    sample_input_data=X_test.head(10),    metrics={"auc_pr": float(auc_pr), "precision_at_80_recall": float(operating_precision)},    comment="Promoted from DEV. Validated on holdout test set.")print("Promoted to STAGING: FRAUD_DEMO_STAGING.ML.FRAUD_DETECTION_MODEL v1")# Promote to PROD (production serving)prod_reg = Registry(session=session, database_name="FRAUD_DEMO_PROD", schema_name="ML")prod_version = prod_reg.log_model(    model=model,    model_name="FRAUD_DETECTION_MODEL",    version_name="v1",    sample_input_data=X_test.head(10),    metrics={"auc_pr": float(auc_pr), "precision_at_80_recall": float(operating_precision)},    comment="Production model. AUC-PR={:.4f}".format(auc_pr))print("Promoted to PROD: FRAUD_DEMO_PROD.ML.FRAUD_DETECTION_MODEL v1")

In [ ]:
# Suspend the training warehouse -- no longer needed until next retrainsession.sql("ALTER WAREHOUSE FRAUD_DEMO_TRAIN_WH SUSPEND").collect()print("\nFRAUD_DEMO_TRAIN_WH suspended. Zero idle cost until next training run.")print(f"\nTotal training cost: ~{train_time/3600 * 6:.2f} credits (${train_time/3600 * 6 * 4.58:.2f})")print("Next retrain: monthly cadence = ~$27/year for training compute.")

---## Summary| What we built | Details ||---|---|| Model | XGBoost binary classifier, 200 trees, max_depth=6 || Training data | 1M rows x 28 features (subset for demo speed; production uses full 12M x 170+) || Class imbalance | scale_pos_weight=2000 (inverse fraud rate) || Primary metric | AUC-PR (appropriate for extreme imbalance) || Training time | ~3-5 min on SP-Optimized MEDIUM (vs hours on SageMaker) || Training cost | ~0.5 credits ($2.29) per run || Registry | Registered in DEV, promoted to STAGING and PROD || Warehouse | SP-Opt MEDIUM (256GB dedicated, 6 credits/hr) -- suspended after use |**Next**: Notebook 4 deploys the model to SPCS for real-time scoring.